# The HBB region in four people

This notebook loads a 6 kb slice of human chromosome 11 (GRCh38) around the beta-globin gene *HBB*, adds the gene, exon and coding-sequence annotations from Ensembl, and applies phased variants from four 1000 Genomes samples. It then draws the result with the reference path laid out as one straight row and every variant branching above or below it.

## Data

The three input files sit next to this notebook and are produced by `prepare_data.py`:

| File | Source | Content |
|------|--------|---------|
| `hbb_region.fa` | Ensembl REST, GRCh38 `chr11:5,224,001-5,230,000` | reference sequence |
| `hbb_region.gff3` | Ensembl REST overlap query | *HBB* gene, MANE Select transcript `HBB-201`, its three exons and coding sequence |
| `hbb_variants.vcf` | 1000 Genomes 30x phased panel (NYGC), GRCh38 | SNVs and small indels for HG00096, NA12878, NA18501 and HG00403 |

The slice is stored as a standalone 6,000 bp sequence named `HBB_region`. A chromosome 11 position `P` therefore appears as `P - 5,224,000` in all three files. *HBB* is on the minus strand, so the gene reads right to left on the chromosome and its promoter is on the right of the slice.

In [ ]:
import pathlib
import tempfile

import gen

DATA_DIR = pathlib.Path(".").resolve()
FASTA = DATA_DIR / "hbb_region.fa"
GFF3 = DATA_DIR / "hbb_region.gff3"
VCF = DATA_DIR / "hbb_variants.vcf"

for path in (FASTA, GFF3, VCF):
    assert path.exists(), f"Missing: {path}"

repo = gen.Repository(tempfile.mkdtemp(prefix="gen-hbb-"))

## Import the reference and the variants

The reference goes in first, then each sample column of the VCF becomes its own sample, derived from the reference.

In [ ]:
repo.import_fasta(str(FASTA), sample="reference")
repo.update_with_vcf(str(VCF), reference="reference")

sequence_graphs = {sg.sample_name: sg for sg in repo.get_sequence_graphs()}
for sample_name, sg in sequence_graphs.items():
    print(f"{sample_name:10s} {sg.name}")

## Reference and one individual

With `center_reference=True` the current path is drawn as one straight row through the middle of the graph. Variant branches leave that row and rejoin it above and below, so the reference stays easy to follow from left to right.

Loading the GFF3 file adds the gene, transcript, exon and coding-sequence features as labelled highlights on the graph.

In [ ]:
sg_reference = sequence_graphs["reference"]

fig_reference = sg_reference.plot(rows=30, center_reference=True, show_history=True)
fig_reference.load_track(str(GFF3), name="HBB")
fig_reference

## A sample with the most variants

NA18501 carries the most alternate alleles of the four samples in this slice (34 alleles, against 20 to 28 for the others). The graph holds the variants of all four samples, and the path of this sample is the one placed on the central row.

In [ ]:
sg_sample = sequence_graphs["NA18501"]

fig_sample = sg_sample.plot(rows=30, center_reference=True, show_history=True)
fig_sample.load_track(str(GFF3), name="HBB")
fig_sample

## Jump to the start codon

`go_to` pans the view to a graph position or search hit. *HBB* is on the minus strand, so its start codon `ATG` is the reverse complement `CAT` at the right-hand end of exon 1. The search below finds a 12 base stretch of the coding sequence a few bases past the start codon, on the reverse strand, and the widget pans to the hit.

In [ ]:
start_codon_hits = sg_sample.search("CTGACTCCTGAG")
print(f"{len(start_codon_hits)} hit(s), strand {start_codon_hits[0].strand}")

fig_sample.go_to(start_codon_hits[0])
fig_sample

In [ ]:
await gen.freeze_all_widgets()